# Formation DTX

[Lien du cours](https://github.com/asardell/data-science-python/blob/main/1-Introduction-data-science.md#chapitre-1--introduction-%C3%A0-la-data-science)

Ce notebook porte sur le sujet des DPE <br>
Lien de l'API : https://data.ademe.fr/datasets/dpe03existant

## Collecte des données

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

### Via l'url en csv



URL pour télécharger les données : https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines?qs=code_departement_ban%3A%2269%22+AND+surface_habitable_logement%3A%3C150&format=csv&size=10000



In [ ]:
#df = pd.read_csv("/content/dpe03existant.csv")

### Via l'API avec Python

In [ ]:
import requests

url = "https://data.ademe.fr/data-fair/api/v1/datasets/meg-83tjwtg8dyz4vv7h1dqe/lines"

params = {
    "qs": "code_departement_ban:69",
    "size": 500
}

toutes_les_lignes = []

while url:

    response = requests.get(url, params=params)
    data = response.json()

    # Ajouter les lignes
    toutes_les_lignes.extend(data["results"])

    print(f"Nombre de lignes : {len(toutes_les_lignes)}")

    # Pour les appels suivants, on utilise directement next
    url = data.get("next")

    # Les paramètres sont déjà présents dans next
    params = None

    if len(toutes_les_lignes) >= 10000:
        break

df = pd.DataFrame(toutes_les_lignes)

print(df.shape)

### Exporter en CSV

Pratique si votre session google colab devient inactive, pensez à télécharger le csv pour éviter de le générer à chaque fois qu'on run tout le notebook

In [ ]:
df.to_csv("dpe03existant.csv")

## Statistiques descriptives

In [ ]:
#afficher un extrait du dataframe
df.head(3)

In [ ]:
# taille du dataframe
df.shape

In [ ]:
# lister les colonnes
sorted(df.columns.to_list())

In [ ]:
# nombre de modalités unique
df["etiquette_dpe"].unique()

In [ ]:
# répartition en effectif
df["etiquette_dpe"].value_counts()

In [ ]:
# répartition en %
df["type_energie_n1"].value_counts(normalize=True)

In [ ]:
# indicateur de position et dispersion
print(df["surface_habitable_logement"].mean())
print(df["surface_habitable_logement"].min())
print(df["surface_habitable_logement"].max())
print(df["surface_habitable_logement"].median())
print(df["surface_habitable_logement"].var())
print(df["surface_habitable_logement"].std())

In [ ]:
# calculer des déciles
df["surface_habitable_logement"].describe(percentiles=[0.1,0.2,
                                                       0.3,0.4,0.5,
                                                       0.6,0.7,0.8,
                                                       0.9])

In [ ]:
# calcul des centiles
import numpy as np
centiles = np.arange(0.01, 1.00, 0.01)
df["surface_habitable_logement"].describe(percentiles=centiles)

In [ ]:
# calcul sur tout le df
df.describe().round(1)

## Graphiques

### Histogramme

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.histplot(df['surface_habitable_logement'], bins = 10)
plt.title("Histogramme de la surface_habitable_logement")
plt.show()

### Boxplot

![Image](https://miro.medium.com/max/9000/1*2c21SkzJMf3frPXPAR_gZA.png)

In [ ]:
# avec seaborn
sns.boxplot(y=df['surface_habitable_logement'], color='lightgreen')
plt.title("Boxplot surface_habitable_logement")
plt.show()

In [ ]:
# avec plotly
import plotly.express as px
fig = px.box(df, y='surface_habitable_logement',
             title="surface_habitable_logement")
fig.show()

print(df["surface_habitable_logement"].describe())

In [ ]:
# boxplot bivarié
fig = px.box(df, x='etiquette_dpe', y='deperditions_murs', title="deperditions_murs par étiquette DPE")
fig.show()
# penser à zoomer avec l'outil plotly

### Diagramme

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ordre_dpe = ["A", "B", "C", "D", "E", "F", "G"]

couleurs_dpe = {
    "A": "#009900",
    "B": "#66CC33",
    "C": "#CCFF00",
    "D": "#FFFF00",
    "E": "#FFCC00",
    "F": "#FF9900",
    "G": "#FF0000"
}

sns.countplot(
    data=df,
    x="etiquette_dpe",
    order=ordre_dpe,
    hue="etiquette_dpe",
    hue_order=ordre_dpe,
    palette=couleurs_dpe,
    legend=False
)

plt.xlabel("Étiquette DPE")
plt.ylabel("Nombre")
plt.title("Répartition des étiquettes DPE")
plt.xticks(rotation=0)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

table = pd.crosstab(
    df["type_energie_n1"],
    df["etiquette_dpe"],
    normalize="index"
) * 100

# Ordre des étiquettes
ordre_dpe = ["A", "B", "C", "D", "E", "F", "G"]

table = table.reindex(columns=ordre_dpe)

# Palette DPE
couleurs = plt.cm.RdYlGn_r(
    [0, 1/6, 2/6, 3/6, 4/6, 5/6, 1]
)

ax = table.plot(
    kind="barh",
    stacked=True,
    figsize=(14, 20),
    color=couleurs
)

ax.set_xlabel("Pourcentage (%)")
ax.set_ylabel("Type d'énergie")
ax.set_title("Étiquette DPE selon le type d'énergie")

ax.legend(
    title="Étiquette DPE",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

couleurs_dpe = {
    "A": "#009900",
    "B": "#66CC33",
    "C": "#CCFF00",
    "D": "#FFFF00",
    "E": "#FFCC00",
    "F": "#FF9900",
    "G": "#FF0000"
}

fig = px.scatter(
    df,
    x="deperditions_murs",
    y="besoin_chauffage",
    color="etiquette_dpe",
    color_discrete_map=couleurs_dpe,
    category_orders={"etiquette_dpe": ["A", "B", "C", "D", "E", "F", "G"]},
    title="Déperditions murs vs besoin chauffage"
)

fig.show()

### Heatmap de corrélation

In [ ]:
ls_columns = [
    'besoin_chauffage',
    'surface_habitable_logement',
    'deperditions_enveloppe',
    'deperditions_ponts_thermiques',
    'deperditions_murs',
    'deperditions_planchers_hauts',
    'deperditions_planchers_bas',
    'deperditions_portes',
    'deperditions_baies_vitrees',
    'deperditions_renouvellement_air'
    ]
corr = df[ls_columns].corr().round(1)
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Heatmap des corrélations")
plt.show()

In [ ]:
import plotly.figure_factory as ff

corr = df[ls_columns].corr().round(1).values
fig = ff.create_annotated_heatmap(corr, x=ls_columns, y=ls_columns, colorscale='RdBu', showscale=True)
fig.show()

## Pandas profiling HTML

il faut installer la librairie qui n'est pas présente par défaut

In [ ]:
pip install ydata_profiling

In [ ]:
from ydata_profiling import ProfileReport

ls_columns = [
 'etiquette_dpe',
 'etiquette_ges',
 'type_batiment',
 'annee_construction',
 'periode_construction',
 'type_installation_chauffage',
 'type_installation_ecs',
 'hauteur_sous_plafond',
 'nombre_appartement',
 'nombre_niveau_immeuble',
 'nombre_niveau_logement',
 'typologie_logement',
 'surface_habitable_logement',
 'classe_inertie_batiment',
 'classe_altitude',
 'zone_climatique',
 'indicateur_confort_ete',
 'protection_solaire_exterieure',
 'logement_traversant',
 'presence_brasseur_air',
 'inertie_lourde',
 'isolation_toiture',
 'deperditions_enveloppe',
 'deperditions_ponts_thermiques',
 'deperditions_murs',
 'deperditions_planchers_hauts',
 'deperditions_planchers_bas',
 'deperditions_portes',
 'deperditions_baies_vitrees',
 'deperditions_renouvellement_air',
 'conso_5_usages_ep',
 'conso_chauffage_ep',
 'conso_ecs_ep',
 'conso_refroidissement_ep',
 'conso_eclairage_ep',
 'conso_auxiliaires_ep',
 'type_energie_n1',
 'cout_total_5_usages',
 'cout_chauffage',
 'cout_ecs',
 'cout_refroidissement',
 'cout_eclairage',
 'cout_auxiliaires',
 'type_energie_principale_chauffage',
 'type_generateur_chauffage_principal',
 'type_installation_chauffage_n1'
 ]

profile = ProfileReport(df[ls_columns],
                        title="Profiling ADEME", explorative=True)


# export en HTML
profile.to_file("profiling_ademe.html")

## Machine Learning

### Classification

#### Choix des colonnes

In [ ]:
# choix des variables explicatives + cible
ls_columns = [
 'etiquette_dpe',
 'periode_construction',
 'hauteur_sous_plafond',
 'surface_habitable_logement',
 'deperditions_enveloppe',
 'deperditions_ponts_thermiques',
 'deperditions_murs',
 'deperditions_planchers_hauts',
 'deperditions_planchers_bas',
 'deperditions_portes',
 'deperditions_baies_vitrees',
 'deperditions_renouvellement_air',
 'type_energie_n1',
 ]

df1 = df[ls_columns]

In [ ]:
df1.shape

#### Gestion des valeurs manquantes

In [ ]:
# gestion des valeurs manquantes
missing_rate = df1.isna().mean().sort_values(ascending=False) * 100
print("Taux de valeurs manquantes (%) par colonne :")
print(missing_rate)

In [ ]:
# remplacement par la médiane ou la modalité "Inconnue"
num_cols = df1.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df1.select_dtypes(include=["object"]).columns

for col in num_cols:
    median_val = df1[col].median()
    df1[col] = df1[col].fillna(median_val)

for col in cat_cols:
    mode_val = df1[col].mode(dropna=True)
    if len(mode_val) > 0:
        df1[col] = df1[col].fillna(mode_val.iloc[0])
    else:
        df1[col] = df1[col].fillna("Inconnu")

print("Imputation terminée.")
print(df1.isna().sum())

#### Création variable cible à prédire

In [ ]:
# Création de la variable cible
df["passoire_energetique"] = df["etiquette_dpe"].isin(["E", "F", "G"])

#### Séparation de X et Y

In [ ]:
Y = df["passoire_energetique"]
X = df.drop(columns=["passoire_energetique", "etiquette_dpe"])

#### Binarisation des variables catégorielles

avec `drop_first=True` pour avoir n-1 colonnes car on peut déduire la dernière (ex: homme/femme, une colonne sufit)

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
# Vérification
X.head(5)

#### Echantillonnage pour séparer le train et le test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.30, random_state=42, stratify=Y
)

print("Train/test split réalisé.")

In [ ]:
print("Taille totale du dataset :", len(X))
print("Taille X_train :", len(X_train))
print("Taille X_test  :", len(X_test))
print("Taille y_train :", len(y_train))
print("Taille y_test  :", len(y_test))

In [ ]:
print("\nDistribution de y_train :")
print(y_train.value_counts(normalize=True))

print("\nDistribution de y_test :")
print(y_test.value_counts(normalize=True))

#### Paramétrage de l'arbre

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_arbre = DecisionTreeClassifier(max_depth=4, min_samples_leaf=30, min_samples_split=50, random_state=42)
model_arbre.fit(X_train, y_train)

#### Interprétation de l'arbre

In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize=(16,10))
plot_tree(model_arbre, feature_names=X.columns, filled=True, fontsize=10)
plt.show()

#### Prédiction sur y_test

In [ ]:
# Prédiction
y_pred = model_arbre.predict(X_test)
y_pred_proba = model_arbre.predict_proba(X_test)

#### Evaluation du modèle

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score

# Matrice de confusion
mc = pd.DataFrame(confusion_matrix(y_test, y_pred),
                     columns=['pred_0','pred_1'],
                     index=['obs_0','obs_1'])
print(mc)

# Métriques
print('Accuracy :', accuracy_score(y_test, y_pred))
print('Recall (binaire) :', recall_score(y_test, y_pred, average='binary', pos_label=0))
print('Precision (binaire) :', precision_score(y_test, y_pred, average='binary', pos_label=0))
print('F1-score (binaire) :', f1_score(y_test, y_pred, average='binary', pos_label=0))

# Micro / Macro / Weighted
print('Recall (macro) :', recall_score(y_test, y_pred, average='macro'))
print('Precision (macro) :', precision_score(y_test, y_pred, average='macro'))
print('F1-score (macro) :', f1_score(y_test, y_pred, average='macro'))

print('Recall (weighted) :', recall_score(y_test, y_pred, average='weighted'))
print('Precision (weighted) :', precision_score(y_test, y_pred, average='weighted'))
print('F1-score (weighted) :', f1_score(y_test, y_pred, average='weighted'))